# Training Supervised Model

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import uniform, randint
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, make_scorer
import joblib

In [2]:
# Load data
train = pd.read_csv('../data/train.csv')
train

,num__network_packet_size,num__login_attempts,num__session_duration,num__ip_reputation_score,num__failed_logins,cat__encryption_used_AES,cat__encryption_used_DES,cat__encryption_used_None,cat__browser_type_Chrome,cat__browser_type_Edge,cat__browser_type_Firefox,cat__protocol_type_TCP,cat__protocol_type_UDP,remainder__unusual_time_access,remainder__attack_detected
0,0.496899,-0.016346,-0.381125,1.554930,-0.500779,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,1
1,-0.143322,-0.525794,0.972960,-0.168029,-1.467959,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0,0
2,0.648132,-0.525794,-0.912503,2.301950,0.466400,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0,1
3,1.530327,-0.016346,-0.243473,-1.174443,-1.467959,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0,1
4,-0.239103,0.493102,-0.330830,-1.560484,-0.500779,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9532,-1.544751,-0.525794,-0.720511,1.052117,1.433580,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0,1
9533,-0.607104,-0.525794,-0.775438,0.435452,-1.467959,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0,0
9534,0.824571,0.493102,-0.963200,0.157266,-0.500779,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0,0
9535,-0.476035,-0.016346,-0.897729,1.163198,-0.500779,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1,0


In [3]:
# Setup pipeline for classification
classifier = Pipeline([
    ('clf', 'passthrough') # Placeholder
])

In [4]:
# Create scorer
f1_scorer = make_scorer(f1_score, average='binary')

In [5]:
# Split data for training and testing
# Acquire target
y = train[['remainder__attack_detected']]
X = train.drop('remainder__attack_detected', axis = 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Setup hyperparameter tuning for multiple models
param_dist = [
    {
        'clf': [LogisticRegression(random_state=42)],
        'clf__solver': ['liblinear', 'lbgfs'],
        'clf__C': uniform(0,1),
        'clf__l1_ratio': uniform(0,1)
    },
    {
        'clf': [RandomForestClassifier(random_state=42)],
        'clf__n_estimators' : randint(1,200),
        'clf__criterion' : ['gini' , 'entropy', 'log_loss'],
        'clf__max_features': ['sqrt', 'log2', None],
        'clf__min_samples_leaf': uniform(1,5)
    },
    {
        'clf': [SVC(random_state=42)],
        'clf__C': uniform(0,1),
        'clf__kernel': ['linear', 'poly', 'rbf'],
        'clf__degree': randint(1,10),
    },
    {
        'clf': [MLPClassifier(random_state=42)],
        'clf__activation': ['relu', 'tanh', 'logistic', 'identity'],
        'clf__solver' : ['lbgfs', 'sgd', 'adam'],
        'clf__alpha': uniform(0,1),
        'clf__learning_rate' : ['constant', 'invscaling', 'adaptive']
    }
    
]

In [7]:
#Build  the random searcher
random_search = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=param_dist,
    cv=5,
    random_state=42,
    scoring=f1_scorer
)

In [8]:
# Do the random search
random_search.fit(X_train,y_train.values.ravel())

/opt/conda/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/sklearn/model_s

RandomizedSearchCV(cv=5, estimator=Pipeline(steps=[('clf', 'passthrough')]),
                   param_distributions=[{'clf': [LogisticRegression(random_state=42)],
                                         'clf__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x767a5d4ddac0>,
                                         'clf__l1_ratio': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x767a5d4dcef0>,
                                         'clf__solver': ['liblinear',...
                                        {'clf': [MLPClassifier(random_state=42)],
                                         'clf__activation': ['relu', 'tanh',
                                                             'logistic',
                                                             'identity'],
                                         'clf__alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x767a69c8f8f0>,
                                         'clf__learning_rate': ['constant',
                                                                'invscaling',
                                                                'adaptive'],
                                         'clf__solver': ['lbgfs', 'sgd',
                                                         'adam']}],
                   random_state=42,
                   scoring=make_scorer(f1_score, response_method='predict', average=binary))

In [9]:
joblib.dump(random_search, '../models/supervised.joblib')

['../models/supervised.joblib']